# Train: Sacral / Disc-Morphology head

Adds a 4th head to `LumbarDiagnosticModel` -- disc morphology
(Normal / Degenerated / Bulging / Herniated / Thinning / Disc
Degeneration with Osteophyte formation) for the lowest 3 disc levels
(L3/L4, L4/L5, L5/S1). L5/S1 is the "sacral" level per the project
brief, since the sacrum is one fused bone with no discs of its own
beyond that junction.

This is a **separate taxonomy** from the existing RSNA severity-grade
head (Normal/Mild, Moderate, Severe) -- both heads coexist on the same
model, trained jointly.

**Data:** Sudirman et al. "Lumbar Spine MRI Dataset" + companion
"Radiologists Notes" (Mendeley Data, **CC BY 4.0** -- free, just cite
the authors):
- Images: https://data.mendeley.com/datasets/k57fr854j2/2
- Notes:  https://data.mendeley.com/datasets/s6bgczr8s2/2

**No Google Drive is used anywhere in this notebook.** Files move
directly between your local machine and this Colab VM's own
(ephemeral, Drive-quota-free) disk.

**If you're running this through the VS Code Colab extension**: its
bridge to the Colab web UI is incomplete -- `google.colab.files.upload()`
and `files.download()` are both currently broken there (known upstream
issues, not something a retry fixes). This notebook uses the documented
workarounds instead: the plain `ipywidgets` File Upload widget for
uploads, and a base64 HTML download link for pulling the trained
checkpoint back. Both are plain Jupyter/browser mechanics, not
Colab-specific JS, so they work the same in VS Code as in the browser.

**Before running the training cell**, run the inspection cell below and
*read its output*. The exact structure of the radiologist-notes file
wasn't confirmed ahead of time -- if it doesn't match what
`SudirmanDiscDataset` expects (a `radiologist_notes.csv` with a
`study_id` column plus one free-text column per level), fix it up in
the "Fix layout if needed" cell before training, not after.

## 1. Runtime check

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'

## 2. Get the project code onto this Colab session

Both the Colab-native `files.upload()` widget and generic `ipywidgets`
render as inert/greyed-out through the VS Code Colab bridge (confirmed
broken, not a fluke -- known upstream limitation). `git clone` is a
plain HTTPS request, no widget/comm channel involved, so it sidesteps
the problem entirely.

The project code (not the datasets, not the full checkpoint) lives at
https://github.com/sameerkulk65/lumbar-mri-dx (public, so no token
needed).

In [ ]:
PROJECT_DIR = '/content/lumbar_mri_dx'
!git clone -q https://github.com/sameerkulk65/lumbar-mri-dx.git {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Get the Sudirman datasets

`_load_image()` only ever reads ONE representative T1-sagittal middle
slice per patient -- the full `MRI_Data.zip` (6.27 GB) is almost
entirely localizer/axial series and other slices that never get used.
Since the Mendeley S3 URL supports HTTP Range requests, the exact 515
files actually needed were pre-extracted directly from the remote
zip's central directory (no full download) and committed to this repo
at `data_cache/sudirman_sagittal_only.zip` (69.8 MB) -- already on
disk from the `git clone` above, no download needed here at all.

The notes file (`Radiologists Report.xlsx`, 38 KB) still comes from
Mendeley directly -- tiny, no issue there. Already verified and fixed:
it's one free-text note per patient (`Patient ID`, `Clinician's Notes`
columns), not the per-level-CSV-column layout originally guessed --
`src/spine_datasets.py` handles this.

In [ ]:
import os, zipfile

SUDIRMAN_DIR = f'{PROJECT_DIR}/data/sudirman'
os.makedirs(f'{SUDIRMAN_DIR}/images', exist_ok=True)

with zipfile.ZipFile(f'{PROJECT_DIR}/data_cache/sudirman_sagittal_only.zip') as zf:
    zf.extractall(f'{SUDIRMAN_DIR}/images')

!wget -q "https://data.mendeley.com/public-files/datasets/s6bgczr8s2/files/ed86d033-5078-4658-948f-94c13e8b2291/file_downloaded" -O "{SUDIRMAN_DIR}/Radiologists Report.xlsx"
print('Extracted to', SUDIRMAN_DIR)

## 5. INSPECT the real file layout -- read this output before continuing

Both sides are already confirmed at this point: notes are
`Radiologists Report.xlsx` (`Patient ID`/`Clinician's Notes` columns,
handled by `split_note_by_level()`), and images are
`images/<batch-folder>/<patient_id>/<session>/<series>/*.ima` --
already matching what `SudirmanDiscDataset._load_image()` expects,
since `data_cache/sudirman_sagittal_only.zip` was pre-extracted with
those exact paths preserved. This cell is now just a quick sanity
check that the extraction landed correctly -- run it and confirm you
see 515 `.ima` files under per-patient sagittal-series folders.

In [ ]:
from pathlib import Path

root = Path(SUDIRMAN_DIR)
print('--- images/ top-level contents (first 40) ---')
img_root = root / 'images'
for p in sorted(img_root.rglob('*'))[:40]:
    print(' ', p.relative_to(img_root))

print('\n--- Sample of one folder, if the top level is per-patient folders ---')
subdirs = [p for p in img_root.iterdir() if p.is_dir()] if img_root.exists() else []
if subdirs:
    sample = subdirs[0]
    print('Folder:', sample.name)
    for f in sorted(sample.iterdir())[:10]:
        print(' ', f.name)
else:
    print('No subfolders directly under images/ -- the real layout differs')
    print('from the <study_id>/*.dcm assumption, inspect the listing above.')

## 5b. Fix layout if needed

If cell 5's output doesn't match `images/<study_id>/*.dcm` (or
`.jpg`), fix it *here* -- either reshape/rename the extracted files to
match, or edit `SudirmanDiscDataset._load_image()` in
`src/spine_datasets.py` directly to match the real layout (e.g. a flat
file-naming convention instead of per-patient folders, or a different
extension). The notes-file side does not need touching.

In [ ]:
# (edit as needed once you've looked at cell 5's output)


## 6. Sanity check: class distribution

Confirms the parser produces a sane, non-degenerate label distribution
before spending GPU hours on it.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)
import numpy as np
from src.spine_datasets import SudirmanDiscDataset, MORPH_LEVELS, MORPH_CLASSES

ds = SudirmanDiscDataset(SUDIRMAN_DIR, split='train')
counts = np.zeros((len(MORPH_LEVELS), len(MORPH_CLASSES)), dtype=int)
for i in range(len(ds)):
    _, target = ds[i]
    for lvl_idx, cls_idx in enumerate(target['morph_labels'].tolist()):
        counts[lvl_idx, cls_idx] += 1

names = list(MORPH_CLASSES.keys()) if isinstance(MORPH_CLASSES, dict) else MORPH_CLASSES
for lvl_idx, level in enumerate(MORPH_LEVELS):
    print(level, dict(zip(names, counts[lvl_idx].tolist())))

print('\nIf one class dominates >90% at every level, the keyword parser')
print('is probably not matching real note phrasing -- go back to cell 5b.')

## 7. Quick CPU-equivalent smoke test (fast on GPU here)

Confirms the model/loss/train wiring is correct on synthetic data
before touching real data or spending a long training run on a bug.

In [ ]:
!python src/train.py --mode smoke

## 7b. Safety net: auto-push checkpoints to GitHub every epoch

Free-tier Colab sessions can disconnect unpredictably (idle timeout,
quota, network) and take everything under `/content/` down with them
-- the repo, the Sudirman download, in-progress checkpoints, all of
it. A manual download link only helps if you're watching and click it
before the disconnect. This replaces that with an automatic push: a
dedicated `checkpoint-latest` branch on your GitHub repo gets
force-pushed after every epoch, so recovery after a disconnect is just
a plain `git clone` on *your local machine* -- no Colab interaction,
no upload widgets, no manual clicking required.

**Before running the next 3 cells**, create a GitHub token scoped to
just this repo:
1. https://github.com/settings/tokens?type=beta -> Generate new token
2. Expiration: short (e.g. 7 days)
3. Repository access -> Only select repositories -> this repo
4. Permissions -> Repository permissions -> Contents -> Read and write
5. Generate, copy the token -- paste it into the cell below

Revoke the token once training is done. Never save/sync this notebook
back to GitHub while the token is pasted in a cell.

In [ ]:
GITHUB_TOKEN = ''  # <-- paste your token here, then clear it before saving this notebook anywhere
assert GITHUB_TOKEN, 'Paste your GitHub token above before continuing.'

In [ ]:
CKPT_PUSH_DIR = '/content/ckpt_push'
CKPT_BRANCH   = 'checkpoint-latest'
REPO_PUSH_URL = f'https://{GITHUB_TOKEN}@github.com/sameerkulk65/lumbar-mri-dx.git'

!rm -rf {CKPT_PUSH_DIR}
!git clone -q --depth 1 -b main "{REPO_PUSH_URL}" {CKPT_PUSH_DIR}
!cd {CKPT_PUSH_DIR} && git checkout -q --orphan {CKPT_BRANCH} && git rm -rf -q .
print('Checkpoint-push clone ready at', CKPT_PUSH_DIR)

In [ ]:
import subprocess, torch
import lightning as L

class GitPushCheckpointCallback(L.Callback):
    """Force-pushes a single amended commit to CKPT_BRANCH after every
    epoch -- the branch always holds just the latest checkpoint, never
    accumulates history/repo size across epochs.

    A push failure (expired token, network blip) must NOT crash the
    training run -- it just means this epoch's checkpoint didn't make it
    to GitHub; the next epoch tries again. Local commit still succeeds
    either way, so `git -C CKPT_PUSH_DIR log`/`diff` can help debug a
    persistent failure without losing anything."""
    def __init__(self, every_n_epochs=1):
        self.every_n_epochs = every_n_epochs
        self._first = True

    def _run(self, *args):
        return subprocess.run(['git', '-C', CKPT_PUSH_DIR] + list(args),
                               capture_output=True, text=True)

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch
        if (epoch + 1) % self.every_n_epochs != 0:
            return
        try:
            state = {'state_dict': pl_module.state_dict(), 'epoch': epoch}
            torch.save(state, f'{CKPT_PUSH_DIR}/latest_checkpoint.pt')

            self._run('add', 'latest_checkpoint.pt')
            commit_args = ['-c', 'user.email=colab@local', '-c', 'user.name=Colab',
                            'commit', '-m', f'checkpoint: epoch {epoch}']
            if not self._first:
                commit_args.append('--amend')
            r = self._run(*commit_args)
            if r.returncode != 0:
                print(f'[checkpoint push] commit failed (epoch {epoch}): ' + r.stderr.strip())
                return

            r = self._run('push', '-q', '--force', 'origin', CKPT_BRANCH)
            if r.returncode != 0:
                print(f'[checkpoint push] push failed (epoch {epoch}), '
                      'checkpoint saved locally only: ' + r.stderr.strip())
                return

            self._first = False
            print(f'Pushed epoch {epoch} checkpoint to {CKPT_BRANCH} branch.')
        except Exception as e:
            print(f'[checkpoint push] unexpected error (epoch {epoch}), continuing training: ' + str(e))

## 8. Train

Initializes encoder/detection/segmentation/classification/morph_head
from `last_state_dict.pt` -- which now already includes the
disc-morphology training from the previous run (epoch 99) -- and
continues training on GPU. Uses `init_from=` (partial `strict=False`
load) so a checkpoint from a slightly different data mix still loads
cleanly.

`skip_spider=True` -- SPIDER's ~9200 slices would otherwise cost
several minutes to download/generate before training even starts and
dominate every batch. `use_rsna_hf=True` brings in
`GilbertKrantz/rsna-lumbar-spine-dataset` (Hugging Face, ungated,
~362MB) -- 30,022 individually-labeled slices covering all 5
conditions x 5 levels, downloaded automatically on first use, no
manual setup needed. This is what actually trains `classification`,
which has been stuck at 11/50 epochs (near-chance confidence) all
project.

**Time budget matters here.** At `rsna_hf_fraction=1.0` (all ~25,500
training slices) epochs would take roughly 75 minutes at the ~86s/489
samples pace seen last run -- likely too long for one Colab session.
`rsna_hf_fraction` below controls how much of it loads; the number
next to each option is the resulting rough per-epoch time so you can
pick based on how long you want this session to run. Default below is
set to the fast end -- more epochs on less data per epoch, closer to
the ~86s pace already seen, rather than fewer epochs on more data.

| fraction | train slices | rough epoch time |
|---|---|---|
| 0.02 (default below) | ~500 | ~2.5 min |
| 0.05 | ~1,300 | ~4 min |
| 0.15 | ~3,800 | ~11 min |
| 0.30 | ~7,700 | ~23 min |
| 1.0 | ~25,500 | ~75 min |

Sudirman's 489 samples ride along in the same run either way -- cheap,
and keeps refining `morph_head` rather than leaving it frozen. A
smaller fraction just means each epoch sees fewer distinct RSNA slices
-- run more epochs (or raise the fraction later) to cover more of it
over time.

Watch this cell's output as it runs -- `Pushed epoch N checkpoint to
checkpoint-latest branch.` should appear after every epoch (see 7b
above). If this session disconnects, that branch always has the
latest completed epoch, recoverable from any machine via git clone.

In [ ]:
import yaml
from src.train import train

with open('configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['data']['skip_spider'] = True
cfg['data']['use_rsna_hf'] = True
cfg['data']['rsna_hf_fraction'] = 0.02  # <-- adjust per the table above

best_ckpt = train(
    cfg,
    init_from=f'{PROJECT_DIR}/outputs/checkpoints/last_state_dict.pt',
    extra_callbacks=[GitPushCheckpointCallback(every_n_epochs=1)],
)
print('Best checkpoint:', best_ckpt)

## 9. Push the final best checkpoint, then recover it locally

Run this once training finishes normally. `best_ckpt` (lowest val/loss,
not necessarily the last epoch trained) gets slimmed down the same way
as the per-epoch pushes and force-pushed to `checkpoint-latest` one
last time, so the branch ends up holding the *best* checkpoint rather
than just whatever the final epoch happened to be.

If training gets cut off instead, skip this cell entirely -- the
branch already has the latest epoch a `GitPushCheckpointCallback` push
succeeded for, and step B below works exactly the same either way.

In [ ]:
import torch

src_ckpt = best_ckpt or f'{PROJECT_DIR}/outputs/checkpoints/last.ckpt'
full = torch.load(src_ckpt, map_location='cpu')
slim = {'state_dict': full['state_dict'], 'epoch': full.get('epoch')}
torch.save(slim, f'{CKPT_PUSH_DIR}/latest_checkpoint.pt')

cb = GitPushCheckpointCallback()
cb._first = False  # always amend -- keep the branch to a single commit
cb._run('add', 'latest_checkpoint.pt')
r = cb._run('-c', 'user.email=colab@local', '-c', 'user.name=Colab',
            'commit', '--amend', '-m', 'checkpoint: final (best)')
if r.returncode != 0:
    print('Commit failed:', r.stderr.strip())
else:
    r = cb._run('push', '-q', '--force', 'origin', CKPT_BRANCH)
    if r.returncode != 0:
        print('Push failed:', r.stderr.strip())
    else:
        print('Pushed final best checkpoint (source:', src_ckpt, ') to', CKPT_BRANCH)

### B. On your LOCAL machine (not in Colab) -- pull it down

No Colab interaction needed for this part -- run in a terminal on
your own machine, any time (mid-training after a disconnect, or after
this final push):

```
git clone -b checkpoint-latest --depth 1 https://github.com/sameerkulk65/lumbar-mri-dx.git recovered_checkpoint
python export_trained_model.py recovered_checkpoint/latest_checkpoint.pt
```

Then restart Streamlit -- the Sacral / Disc Morphology section will
show real predictions instead of untrained ones.